# Лабораторная работа 17. Суперпозиция и разреженные автоэнкодеры (студенческая версия)

**Курс:** Методы обработки естественного языка · **Блок 6:** интерпретируемость и in-context learning

**Цель.** Воспроизвести явление суперпозиции на игрушечной модели и обучить sparse autoencoder (SAE) на активациях GPT-2, чтобы извлечь интерпретируемые признаки и использовать их для управления генерацией.

**Что нужно сдать:**
1. Игрушечную модель суперпозиции (Elhage et al., 2022): зависимость числа представленных признаков от разреженности.
2. Сбор активаций residual stream GPT-2 small.
3. SAE с L1-штрафом и TopK SAE; метрики: L0, доля объяснённой дисперсии, мёртвые признаки, spliced loss.
4. Интерпретацию 20 признаков по топ-активирующим контекстам.
5. Steering: добавление направления признака в residual stream.

**Теория.**
- **Суперпозиция:** модель с \(m\) нейронами представляет \(n > m\) признаков почти ортогональными направлениями. Это выгодно, когда признаки разрежены: интерференция редко проявляется одновременно.
- **SAE:** \(f(x) = \mathrm{ReLU}(W_e(x - b_d) + b_e)\), \(\hat x = W_d f(x) + b_d\), столбцы \(W_d\) нормированы. L1-вариант: \(\mathcal{L} = \|x - \hat x\|^2 + \lambda\|f\|_1\). TopK-вариант (Gao et al., 2024): оставить \(k\) наибольших предактиваций, \(\mathcal{L} = \|x - \hat x\|^2\).
- **Spliced loss:** CE языковой модели при подмене активаций слоя их реконструкцией; доля восстановленного loss \(= \frac{CE_{zero} - CE_{sae}}{CE_{zero} - CE_{clean}}\).

In [ ]:
# ==== Общая настройка (одинакова во всех работах) ====
import os, math, time, json, random, pathlib, urllib.request
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

FAST_DEV_RUN = True          # True — сокращённый прогон (CPU, минуты); False — полный эксперимент
SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
ARTIFACTS = pathlib.Path("artifacts"); ARTIFACTS.mkdir(exist_ok=True)
DATA = pathlib.Path("data"); DATA.mkdir(exist_ok=True)

def set_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
set_seed()

def download(url, name):
    path = DATA / name
    if not path.exists():
        urllib.request.urlretrieve(url, path)
    return path

def load_shakespeare():
    url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
    return download(url, "shakespeare.txt").read_text(encoding="utf-8")

def load_russian():
    """Русский корпус: «Война и мир» (Л. Н. Толстой), т. 1, из открытого репозитория.
    При недоступности сети используется встроенный фрагмент."""
    urls = ["https://raw.githubusercontent.com/mmcdermott/nlp-war-and-peace/master/data/war_and_peace_ru.txt",
            "https://raw.githubusercontent.com/nevmenandr/word2vec-russian-novels/master/books_before/WarAndPeace.txt"]
    for i, u in enumerate(urls):
        try:
            t = download(u, f"ru_{i}.txt").read_text(encoding="utf-8", errors="ignore")
            if len(t) > 100_000: return t
        except Exception:
            pass
    base = ("Все счастливые семьи похожи друг на друга, каждая несчастливая семья несчастлива по-своему. "
            "Всё смешалось в доме Облонских. Жена узнала, что муж был в связи с бывшею в их доме француженкою-гувернанткой, "
            "и объявила мужу, что не может жить с ним в одном доме. ")
    return base * 400

print(f"torch {torch.__version__}, device={DEVICE}, FAST_DEV_RUN={FAST_DEV_RUN}")

## 1. Игрушечная модель суперпозиции

\(n = 20\) признаков, \(m = 5\) скрытых измерений. Признак \(x_i\) равен нулю с вероятностью \(S\), иначе \(\sim U[0,1]\). Важность \(I_i = 0.7^i\). Модель: \(\hat x = \mathrm{ReLU}(W^\top W x + b)\), \(W \in \mathbb{R}^{m\times n}\), loss \(\sum_i I_i (x_i - \hat x_i)^2\).

Для \(S \in \{0, 0.7, 0.9, 0.97, 0.99\}\) обучите модель, постройте \(W^\top W\) и нормы \(\|W_i\|\). Посчитайте число представленных признаков (\(\|W_i\| > 0.5\)) и долю «суперпозиции» для каждого признака: \(\sum_{j\ne i}(\hat W_i\cdot W_j)^2 / \|W_i\|^2\).

In [ ]:
def train_toy(S, n=20, m=5, steps=4000, B=1024, lr=1e-2, seed=0):
    g = torch.Generator().manual_seed(seed)
    W = (torch.randn(m, n, generator=g) * 0.1).requires_grad_(); b = torch.zeros(n, requires_grad=True)
    I = 0.7 ** torch.arange(n).float(); opt = torch.optim.Adam([W, b], lr=lr)
    # TODO: ваш код
    raise NotImplementedError
    return W.detach(), b.detach()

SPARS = [0.0, 0.7, 0.9, 0.97, 0.99]
fig, axes = plt.subplots(2, len(SPARS), figsize=(16, 6))
for j, S in enumerate(SPARS):
    W, b = train_toy(S, steps=2000 if FAST_DEV_RUN else 10000)
    norms = W.norm(dim=0); WhatT = W / norms.clamp(min=1e-6)
    sup = ((WhatT.T @ W) ** 2).sum(1) - norms ** 2
    sup = sup / (norms ** 2).clamp(min=1e-6)
    axes[0, j].imshow((W.T @ W).numpy(), cmap="RdBu", vmin=-1, vmax=1); axes[0, j].set_title(f"S={S}: WᵀW")
    axes[1, j].barh(range(20), norms.numpy(), color=plt.cm.viridis(sup.clamp(0, 1).numpy())); axes[1, j].invert_yaxis()
    axes[1, j].set_title(f"признаков: {(norms > 0.5).sum().item()}"); axes[1, j].set_xlim(0, 1.3)
plt.tight_layout(); plt.show()

## 2. Активации GPT-2

Соберите residual stream после блока `LAYER` = 6 (вход блока 7) на новостных текстах AG News. Исключите позицию 0: у GPT-2 её норма аномально большая (attention sink). Нормируйте активации так, чтобы средняя норма была \(\sqrt{d}\). Для экономии времени прерывайте прямой проход после нужного слоя исключением в хуке.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import pandas as pd
tok = AutoTokenizer.from_pretrained("gpt2"); gpt = AutoModelForCausalLM.from_pretrained("gpt2").to(DEVICE).eval()
for p in gpt.parameters(): p.requires_grad_(False)
LAYER, CTX = 6, 64
path = download("https://raw.githubusercontent.com/mhjabreel/CharCnn_Keras/master/data/ag_news_csv/train.csv", "ag_train.csv")
news = pd.read_csv(path, header=None, names=["y", "title", "desc"])
texts = (news.title + ". " + news.desc).str.replace("\\\\", " ", regex=False).tolist()
N_TOKENS = 60_000 if FAST_DEV_RUN else 1_000_000

class Stop(Exception): pass
def residual(ids):
    store = {}
    def hook(mod, inp, out):
        store["x"] = (out[0] if isinstance(out, tuple) else out).detach(); raise Stop
    h = gpt.transformer.h[LAYER].register_forward_hook(hook)
    try:
        with torch.no_grad(): gpt(ids)
    except Stop: pass
    finally: h.remove()
    return store["x"]

# TODO: ваш код (токенизировать тексты в окна по CTX токенов, собрать активации (без позиции 0), сохранить контексты для интерпретации)

## 3. SAE: L1 и TopK

Реализуйте SAE с:
- инициализацией \(W_e = W_d^\top\) и нормированными столбцами \(W_d\), \(b_d\) = среднему активаций;
- нормировкой столбцов \(W_d\) после каждого шага оптимизатора;
- режимами `l1` (штраф \(\lambda\|f\|_1\)) и `topk`.

Обучите оба варианта (размер словаря \(8d\), в `FAST_DEV_RUN` — \(4d\)). Подберите \(\lambda\) так, чтобы L0 у L1-варианта было близко к \(k\) TopK-варианта (\(k = 32\)). Посчитайте на отложенных 10% активаций: L0, FVU \(= \|x - \hat x\|^2 / \|x - \bar x\|^2\), долю мёртвых признаков (ни разу не активны).

In [ ]:
class SAE(nn.Module):
    def __init__(self, d, n_feat, mode="topk", k=32, l1=1e-3, b_dec=None):
        super().__init__()
        self.mode, self.k, self.l1 = mode, k, l1
        Wd = torch.randn(d, n_feat); Wd = Wd / Wd.norm(dim=0, keepdim=True)
        self.W_dec = nn.Parameter(Wd); self.W_enc = nn.Parameter(Wd.T.clone())
        self.b_enc = nn.Parameter(torch.zeros(n_feat)); self.b_dec = nn.Parameter(b_dec.clone() if b_dec is not None else torch.zeros(d))
    def encode(self, x):
        # TODO: ваш код
        raise NotImplementedError
    def decode(self, f): return f @ self.W_dec.T + self.b_dec
    def forward(self, x):
        f = self.encode(x); xh = self.decode(f)
        # TODO: ваш код
        raise NotImplementedError
        return xh, f, loss
    @torch.no_grad()
    def normalize(self): self.W_dec.data /= self.W_dec.data.norm(dim=0, keepdim=True)

perm = torch.randperm(len(X), generator=torch.Generator().manual_seed(0)); n_tr = int(0.9 * len(X))
X_tr, X_te = X[perm[:n_tr]], X[perm[n_tr:]]
N_FEAT = 4 * X.size(1) if FAST_DEV_RUN else 8 * X.size(1)

def train_sae(mode, epochs=3 if FAST_DEV_RUN else 10, B=512, lr=1e-3, **kw):
    set_seed(0); sae = SAE(X.size(1), N_FEAT, mode, b_dec=X_tr.mean(0), **kw).to(DEVICE); opt = torch.optim.Adam(sae.parameters(), lr=lr)
    for ep in range(epochs):
        for i in torch.randperm(len(X_tr)).split(B):
            _, _, loss = sae(X_tr[i].to(DEVICE)); opt.zero_grad(); loss.backward(); opt.step(); sae.normalize()
    return sae.eval()

@torch.no_grad()
def sae_metrics(sae, Xe=None):
    Xe = (X_te if Xe is None else Xe).to(DEVICE); xh, f, _ = sae(Xe)
    return {"L0": (f > 0).float().sum(1).mean().item(), "FVU": ((Xe - xh).pow(2).sum() / (Xe - Xe.mean(0)).pow(2).sum()).item(),
            "мёртвых, %": 100 * ((f > 0).sum(0) == 0).float().mean().item()}

t0 = time.time(); sae_topk = train_sae("topk", k=32); print("TopK:", sae_metrics(sae_topk), f"{time.time() - t0:.0f} c")
# TODO: ваш код (подбор λ для L1-SAE (2–3 значения) и таблица метрик)

### Spliced loss

Подмените выход блока `LAYER` реконструкцией SAE (в исходном масштабе, позицию 0 оставьте без изменений) и посчитайте CE GPT-2 на 50 окнах. Сравните с чистым прогоном и с zero-ablation (residual после слоя обнулён, кроме позиции 0). Вычислите долю восстановленного loss.

In [ ]:
@torch.no_grad()
def lm_loss(ids, replace=None):
    def hook(mod, inp, out):
        hs = out[0] if isinstance(out, tuple) else out
        hs = hs.clone(); hs[:, 1:] = replace(hs[:, 1:])
        return (hs,) + tuple(out[1:]) if isinstance(out, tuple) else hs
    h = gpt.transformer.h[LAYER].register_forward_hook(hook) if replace else None
    try: return gpt(ids, labels=ids).loss.item()
    finally:
        if h: h.remove()

eval_ids = windows[-50:].to(DEVICE)
# TODO: ваш код (CE clean, zero, SAE TopK, SAE L1 и доля восстановленного loss)

## 4. Интерпретация признаков

Для TopK SAE выберите 20 случайных живых признаков (частота срабатывания от 0.01% до 5%). Для каждого выведите 5 контекстов с наибольшей активацией: 8 токенов слева, активирующий токен в скобках, 2 токена справа. Предложите краткое описание каждого признака и оцените свою уверенность (уверен / частично / не понятно). Посчитайте долю признаков с понятной интерпретацией.

In [ ]:
@torch.no_grad()
def feature_freq(sae, bs=4096):
    return sum((sae.encode(X[i:i + bs].to(DEVICE)) > 0).float().sum(0).cpu() for i in range(0, len(X), bs)) / len(X)

@torch.no_grad()
def feature_acts(sae, feats, bs=4096):
    """Активации выбранных признаков на всех токенах: (N, len(feats)) — полная матрица не помещается в память."""
    return torch.cat([sae.encode(X[i:i + bs].to(DEVICE))[:, feats].cpu() for i in range(0, len(X), bs)])

# TODO: ваш код (частоты, выбор 20 признаков, вывод топ-контекстов)

## 5. Steering

Выберите признак, связанный с темой (например, спорт: найдите признак с наибольшей средней активацией на токенах « football», « game», « team» в коротких промптах). Исключите признаки с частотой выше 5%: они срабатывают почти везде и не несут темы. Добавьте \(\alpha\cdot\text{SCALE}\cdot W_d[:, j]\) к residual stream после слоя `LAYER` на всех позициях при генерации и сравните продолжения при \(\alpha \in \{0, 10, 20, 40\}\) для нейтрального промпта. Для ориентира: средняя норма residual stream равна \(\sqrt{d}\cdot\text{SCALE} \approx 90\).

In [ ]:
# TODO: ваш код (выбор признака по промптам, хук steering, генерация при разных α)

## Вопросы

1. Почему суперпозиция появляется только при высокой разреженности? Объясните фазовый переход на игрушечной модели через баланс выгоды представления признака и стоимости интерференции.
2. Почему у L1-SAE возникает «усадка» (shrinkage) активаций и как TopK её устраняет? Какие проблемы есть у TopK?
3. Что означает низкая FVU при плохом spliced loss? Почему важны обе метрики?
4. Откуда берутся мёртвые признаки и как с ними борются (resampling, aux-loss в TopK, инициализация)?
5. Насколько устойчиво steering и почему слишком большое \(\alpha\) разрушает текст?